# Глава 10 — Code Agents and Code LLMs

Coding agent соединяет модель с инструментами, которые позволяют **читать код, менять файлы, выполнять программу и использовать её результат в следующем шаге**. Это тот же цикл ReAct, но теперь среда — рабочая папка и Python-процесс.

Добавлены `CodeWorkspace`, четыре кодовых инструмента, `Display` и `cli.py`. Базовый `TinyAgent` остаётся совместим с предыдущими главами; отображение событий включается явно.

**Граница исполнения:** `execute_python` запускает локальный Python с правами пользователя. Рабочая папка и тайм-аут не делают его песочницей. В обычном CLI запись и запуск кода требуют подтверждения. В этом notebook временная папка и callback разрешают только показанный ниже конкретный файл и конкретный код; все другие изменения и запуски отклоняются.

In [ ]:
from pathlib import Path
import socket
from tempfile import TemporaryDirectory

from agent import TinyAgent
from display import Display
from llm import LLM, Response
from memory import Memory
from planning import NativeReAct
from toolbox import CodeWorkspace, make_code_tools
from illustrated_agents.utils import TrajectoryViewer

socket.setdefaulttimeout(180)
llm = LLM("gemma4:e4b", think=True, temperature=0)

# Keep the owner alive until the final cleanup cell.
workspace_owner = TemporaryDirectory(prefix="tinyagent-chapter10-")
workspace_path = Path(workspace_owner.name)
workspace = CodeWorkspace(str(workspace_path))
(workspace_path / "task.txt").write_text(
    "We sell 5000 units monthly at 29 dollars each, with a unit cost of 14 dollars. "
    "A 10% price cut raises volume by 10%. Find the change in monthly profit.",
    encoding="utf-8",
)
print("Temporary workspace:", workspace.root)

## Инструменты и подтверждение

`read_file` и `list_files` читают выбранную рабочую папку. `write_file` заменяет содержимое указанного файла, а `execute_python` запускает код. Файловые инструменты отклоняют пути и symlink, ведущие за пределы рабочей папки; произвольный Python этим ограничением не связан.

Для воспроизводимого запуска разрешаем только точный текст расчёта ниже. Это проверка интеграции чтения, записи, выполнения и observations, а не самостоятельного программирования моделью.

In [ ]:
APPROVED_CODE = """original_profit = 5000 * (29 - 14)
new_profit = (5000 * 1.10) * (29 * 0.90 - 14)
change = new_profit - original_profit
print(f"original_profit={original_profit:.2f}")
print(f"new_profit={new_profit:.2f}")
print(f"change={change:.2f}")
"""
print(APPROVED_CODE)
approval_requests = []

def approve_example(name, kwargs):
    allowed = (
        name == "write_file"
        and set(kwargs) == {"path", "content"}
        and kwargs["path"] == "profit.py"
        and kwargs["content"].strip() == APPROVED_CODE.strip()
    ) or (
        name == "execute_python"
        and set(kwargs) == {"code"}
        and kwargs["code"].strip() == APPROVED_CODE.strip()
    )
    approval_requests.append({"tool": name, "allowed": allowed})
    return allowed

tools = make_code_tools(workspace, approval=approve_example)
agent = TinyAgent(llm, Memory(), tools, NativeReAct(max_steps=8), display=Display(color=False))
print("Registered:", list(tools.registry))

## 1. Модель читает задачу

События `thinking`, `response`, `tool_call`, `observation` выводятся по мере выполнения. `THOUGHT` отображает reasoning, если модель его вернула; `ANSWER` появляется только для окончательного ответа, а не промежуточного сообщения с вызовом инструмента.

In [ ]:
answer = agent.run("Use list_files to inspect the workspace, then read task.txt with read_file and briefly summarize the requested calculation. Do not write files or execute code yet.")
read_steps = agent.trajectory.runs[-1]["steps"]
read_actions = [step.action["tool"] for step in read_steps if step.action]
assert "list_files" in read_actions and "read_file" in read_actions
assert not approval_requests
assert sorted(p.name for p in workspace_path.iterdir()) == ["task.txt"]

## 2. Модель сохраняет и запускает согласованный код

Просим записать `profit.py` и передать **тот же исходный текст** в `execute_python`. Callback не разрешает альтернативный код, импорты или запуск произвольных файлов.

Расчёт из главы: исходная прибыль — $75 000; новая — $66 550; изменение — **−$8 450**. Это арифметическая модель с неизменной себестоимостью единицы, без прочих расходов.

In [ ]:
task = (
    "Use write_file to create profit.py with the exact code below. "
    "Then use execute_python with that exact code as its code argument (not an import or a file runner). "
    "Do not change the code: only this exact snippet is approved. "
    "Use the actual output to explain the change in monthly profit.\n\n" + APPROVED_CODE
)
answer = agent.run(task)
steps = agent.trajectory.runs[-1]["steps"]
write_steps = [step for step in steps if step.action and step.action["tool"] == "write_file"]
exec_steps = [step for step in steps if step.action and step.action["tool"] == "execute_python"]
assert write_steps and exec_steps
assert (workspace_path / "profit.py").read_text().strip() == APPROVED_CODE.strip()
successful = [step for step in exec_steps if "original_profit=75000.00" in step.observation]
assert successful
assert "new_profit=66550.00" in successful[-1].observation
assert "change=-8450.00" in successful[-1].observation
assert any(item == {"tool": "write_file", "allowed": True} for item in approval_requests)
assert any(item == {"tool": "execute_python", "allowed": True} for item in approval_requests)
assert steps[-1].observation is None and steps[-1].answer == answer
assert "8450" in answer.replace(",", "").replace(" ", "")
print("Verified file, execution output and final answer.")
print("Approval decisions:", approval_requests)

In [ ]:
TrajectoryViewer(agent.trajectory)

## 3. Отказ — тоже observation

Этот пример не обращается к модели и ничего не запускает. Он демонстрирует стандартную точку контроля перед выполнением. Агент может получить такой результат и объяснить отказ или предложить другое действие.

In [ ]:
denied_tools = make_code_tools(workspace, approval=lambda name, kwargs: False)
result = denied_tools.execute(Response(tool_call={
    "tool": "execute_python", "kwargs": {"code": "print('not executed')"},
}))
print(result)
assert "denied" in result
assert "not executed" not in result

## CLI: чат в терминале

Из корня проекта создайте отдельную рабочую папку и запустите:

```sh
mkdir -p /tmp/tinyagent-playground
.venv/bin/python cli.py --workspace /tmp/tinyagent-playground
```

Например: «Создай Python-функцию сложения в calculator.py и проверь её». При `write_file` и `execute_python` будут показаны **имя и аргументы** с запросом `[y/N]`. Подтвердите только конкретное действие, которое просмотрели. Пустой ответ означает отказ. `exit`, `quit`, Ctrl-D или Ctrl-C завершают чат.

Параметры: `--model`, `--base-url`, `--max-steps`, `--timeout`, `--no-color`. Ключ при необходимости задаётся переменной `TINYAGENT_API_KEY`. Нет режима автоматического разрешения всех записей и запусков.

Без `display` библиотечный TinyAgent работает без вывода событий, как раньше. Для текстового ReAct и мультимодальных запросов отображение также подключается через `display=Display()`.

## Границы реализации и теория главы

- Выполнение использует текущий Python и его окружение. Это не контейнер: нет изоляции сети, секретов, файлов вне папки, памяти и CPU.
- Тайм-аут по умолчанию — 30 секунд. На POSIX при тайм-ауте завершается группа процессов; на других платформах — основной процесс. Это не защита от намеренно обходящего ограничения кода.
- Ответы инструментов ограничиваются `max_output` символами с явным маркером обрезки. stdout/stderr сначала собираются в памяти: это лимит отображения, а не лимит памяти процесса.
- Ошибки Python возвращаются с exit code, stdout и stderr; отказ и тайм-аут также становятся observations. Они не считаются успешным выполнением задачи.
- В notebook разрешён только фиксированный пример. Он не проверяет качество генерации произвольного кода или исправления реальных GitHub issues.

Hosted code execution на Gemini, репозиторные карты, поиск по коду, SQL, SWE-bench, Agentless и обучение coding LLM обсуждаются в книге, но не входят в этот локальный пример. Модель здесь не обучается, внешние платные API не подключаются.

In [ ]:
workspace_owner.cleanup()
print("Temporary workspace removed; the agent trajectory is still available in memory.")